## Setup

In [14]:
import pandas as pd

In [18]:
df=pd.read_csv('../data/cville_weather_cleaned.csv')

In [19]:
print(df.head)

<bound method NDFrame.head of           STATION        DATE                        NAME  PRCP  SNOW  SNWD  \
0     USC00441593  2000-01-01  CHARLOTTESVILLE 2 W, VA US   0.0   0.0   0.0   
1     USC00441593  2000-01-02  CHARLOTTESVILLE 2 W, VA US   0.0   0.0   0.0   
2     USC00441593  2000-01-03  CHARLOTTESVILLE 2 W, VA US   0.0   0.0   0.0   
3     USC00441593  2000-01-04  CHARLOTTESVILLE 2 W, VA US   0.0   0.0   0.0   
4     USC00441593  2000-01-05  CHARLOTTESVILLE 2 W, VA US  86.0   0.0   0.0   
...           ...         ...                         ...   ...   ...   ...   
9418  USC00441593  2025-12-13  CHARLOTTESVILLE 2 W, VA US   0.0   0.0   0.0   
9419  USC00441593  2025-12-14  CHARLOTTESVILLE 2 W, VA US  13.0   5.0   0.0   
9420  USC00441593  2025-12-15  CHARLOTTESVILLE 2 W, VA US   0.0   0.0   0.0   
9421  USC00441593  2025-12-16  CHARLOTTESVILLE 2 W, VA US   0.0   0.0   0.0   
9422  USC00441593  2025-12-17  CHARLOTTESVILLE 2 W, VA US   0.0   0.0   0.0   

      TMAX  TMIN  WT0

In [20]:
print(df.describe())

              PRCP         SNOW         SNWD         TMAX         TMIN  \
count  9423.000000  9423.000000  9423.000000  9423.000000  9423.000000   
mean     32.919134     1.142948     4.158867    67.765478    48.120970   
std      93.474705    12.490695    26.401958    17.618425    15.943042   
min       0.000000     0.000000     0.000000    17.100000    -0.900000   
25%       0.000000     0.000000     0.000000    54.000000    35.100000   
50%       0.000000     0.000000     0.000000    70.000000    48.900000   
75%      13.000000     0.000000     0.000000    82.900000    62.100000   
max    1765.000000   419.000000   483.000000   105.100000    80.100000   

              WT01         WT03         WT04         WT05         WT06  \
count  9423.000000  9423.000000  9423.000000  9423.000000  9423.000000   
mean      0.175846     0.082670     0.009339     0.001061     0.006049   
std       0.380710     0.275397     0.096191     0.032561     0.077544   
min       0.000000     0.000000     0

## EDA

In [29]:
weather_counts_df = df.copy()
weather_counts_df["DATE"] = pd.to_datetime(weather_counts_df["DATE"])
weather_counts_df["year"] = weather_counts_df["DATE"].dt.year

transition_matrix = pd.crosstab(
    df["state"], 
    df["next_state"], 
    normalize="index"
)

state_order = list(transition_matrix.index)

# aggregate counts
aggregate_counts = (
    weather_counts_df["state"]
    .value_counts()
    .reindex(state_order, fill_value=0)
    .rename_axis("state")
    .reset_index(name="days")
)

# counts by year
counts_by_year = (
    pd.crosstab(weather_counts_df["year"], weather_counts_df["state"])
    .reindex(columns=state_order, fill_value=0)
    .sort_index()
)

# counts by season
counts_by_season = (
    pd.crosstab(weather_counts_df["season"], weather_counts_df["state"])
    .reindex(columns=state_order, fill_value=0)
)

season_order = ["Winter", "Spring", "Summer", "Fall"]
ordered_present_seasons = [s for s in season_order if s in counts_by_season.index]
remaining_seasons = [s for s in counts_by_season.index if s not in ordered_present_seasons]
counts_by_season = counts_by_season.loc[ordered_present_seasons + remaining_seasons]

print("AGGREGATE COUNTS")
display(aggregate_counts)

print("COUNTS BY YEAR")
display(counts_by_year)

print("COUNTS BY SEASON")
display(counts_by_season)

AGGREGATE COUNTS


,state,days
0,Cold-Dry,3509
1,Cold-Rainy,472
2,Fog,239
3,Heavy-Rain,1706
4,High Winds,52
5,Hot-Dry,1324
6,Hot-Rainy,149
7,Ice,57
8,Mild-Dry,852
9,Mild-Rainy,211


COUNTS BY YEAR


state,Cold-Dry,Cold-Rainy,Fog,Heavy-Rain,High Winds,Hot-Dry,Hot-Rainy,Ice,Mild-Dry,Mild-Rainy,Sleet,Thunder
year,,,,,,,,,,,,
2000,138,14,0,71,0,49,12,1,39,11,1,0
2001,155,25,1,67,0,57,15,0,33,10,0,2
2002,147,21,0,82,0,80,5,0,20,7,0,3
2003,129,25,2,114,1,32,6,0,29,9,2,16
2004,134,24,7,82,1,34,9,0,41,7,2,25
2005,136,20,1,84,1,51,7,2,39,4,2,18
2006,145,24,7,68,4,35,6,0,28,14,1,33
2007,131,16,17,60,9,53,5,0,35,6,3,30
2008,116,18,11,62,4,53,9,4,26,6,2,25


COUNTS BY SEASON


state,Cold-Dry,Cold-Rainy,Fog,Heavy-Rain,High Winds,Hot-Dry,Hot-Rainy,Ice,Mild-Dry,Mild-Rainy,Sleet,Thunder
season,,,,,,,,,,,,
Winter,1478,195,64,444,26,0,0,51,7,3,54,11
Spring,1008,167,53,481,16,184,17,2,182,40,15,197
Summer,14,1,40,362,1,917,115,0,353,96,1,492
Fall,1009,109,82,419,9,223,17,4,310,72,4,78


In [34]:
import numpy as np
import matplotlib.pyplot as plt

eda_df = df.copy()
eda_df["DATE"] = pd.to_datetime(eda_df["DATE"])
eda_df = eda_df.sort_values("DATE").reset_index(drop=True)
eda_df = eda_df.dropna(subset=["state"]).copy()

state_order = list(transition_matrix.index)

transition_counts = pd.crosstab(eda_df["state"], eda_df["next_state"]).reindex(
    index=state_order,
    columns=list(transition_matrix.columns),
    fill_value=0
)

# persistence probability
common_states = [s for s in transition_matrix.index if s in transition_matrix.columns]

self_transition_probs = pd.DataFrame({
    "state": common_states,
    "self_transition_probability": [transition_matrix.loc[s, s] for s in common_states]
}).sort_values("self_transition_probability", ascending=False)

print("SELF-TRANSITION (PERSISTENCE) PROBABILITIES")
display(self_transition_probs.round(4))

# most common changes in state
off_diag = transition_counts.copy()

for s in off_diag.index:
    if s in off_diag.columns:
        off_diag.loc[s, s] = 0

common_transitions = (
    off_diag.stack()
    .reset_index()
    .rename(columns={"state": "from_state", "next_state": "to_state", 0: "count"})
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print("MOST COMMON NON-SELF TRANSITIONS")
display(common_transitions.head(15))

# how long each state persists
eda_df["state_block"] = (eda_df["state"] != eda_df["state"].shift()).cumsum()

run_lengths = (
    eda_df.groupby(["state_block", "state"])
    .size()
    .reset_index(name="run_length_days")
)

run_length_summary = (
    run_lengths.groupby("state")["run_length_days"]
    .agg(["count", "mean", "median", "max"])
    .sort_values("mean", ascending=False)
)

print("RUN LENGTH SUMMARY BY STATE")
display(run_length_summary.round(2))

# probabilities of changes in state
row_totals = transition_counts.sum(axis=1)

sudden_transition_table = common_transitions.copy()
sudden_transition_table["probability_given_from_state"] = sudden_transition_table.apply(
    lambda r: (
        r["count"] / row_totals.loc[r["from_state"]]
        if row_totals.loc[r["from_state"]] > 0 else np.nan
    ),
    axis=1
)

print("NON-SELF TRANSITIONS WITH CONDITIONAL PROBABILITIES")
display(sudden_transition_table.head(20).round(4))

# baby plots

# state frequencies
plt.figure(figsize=(10, 5))
eda_df["state"].value_counts().reindex(state_order).plot(kind="bar")
plt.title("Weather State Frequencies")
plt.xlabel("State")
plt.ylabel("Number of Days")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../outputs/EDA/weather_state_freqeuncies.png")
plt.close()

# self-transition probabilities
plt.figure(figsize=(10, 5))
self_transition_probs.set_index("state")["self_transition_probability"].plot(kind="bar")
plt.title("Self-Transition Probability by State")
plt.xlabel("State")
plt.ylabel("Probability of Remaining in Same State")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../outputs/EDA/self_transition_probability.png")
plt.close()


# average run length by state
plt.figure(figsize=(10, 5))
run_length_summary["mean"].reindex(state_order).plot(kind="bar")
plt.title("Average Run Length by State")
plt.xlabel("State")
plt.ylabel("Average Consecutive Days")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../outputs/EDA/average_run_length.png")
plt.close()

SELF-TRANSITION (PERSISTENCE) PROBABILITIES


,state,self_transition_probability
0,Cold-Dry,0.7062
5,Hot-Dry,0.5642
8,Mild-Dry,0.3908
3,Heavy-Rain,0.3206
11,Thunder,0.3072
10,Sleet,0.2432
7,Ice,0.2105
2,Fog,0.1172
9,Mild-Rainy,0.1137
1,Cold-Rainy,0.0911


MOST COMMON NON-SELF TRANSITIONS


,from_state,to_state,count
0,Heavy-Rain,Cold-Dry,481
1,Cold-Dry,Heavy-Rain,439
2,Cold-Rainy,Cold-Dry,267
3,Hot-Dry,Thunder,230
4,Mild-Dry,Hot-Dry,210
5,Cold-Dry,Cold-Rainy,201
6,Thunder,Heavy-Rain,192
7,Heavy-Rain,Cold-Rainy,173
8,Cold-Dry,Mild-Dry,139
9,Thunder,Hot-Dry,138


RUN LENGTH SUMMARY BY STATE


,count,mean,median,max
state,,,,
Cold-Dry,1032,3.40,3.0,20
Hot-Dry,577,2.29,2.0,13
Mild-Dry,519,1.64,1.0,8
Heavy-Rain,1159,1.47,1.0,10
Thunder,539,1.44,1.0,6
Sleet,56,1.32,1.0,3
Ice,45,1.27,1.0,3
Fog,211,1.13,1.0,4
Mild-Rainy,187,1.13,1.0,3


NON-SELF TRANSITIONS WITH CONDITIONAL PROBABILITIES


,from_state,to_state,count,probability_given_from_state
0,Heavy-Rain,Cold-Dry,481,0.2819
1,Cold-Dry,Heavy-Rain,439,0.1251
2,Cold-Rainy,Cold-Dry,267,0.5657
3,Hot-Dry,Thunder,230,0.1737
4,Mild-Dry,Hot-Dry,210,0.2465
5,Cold-Dry,Cold-Rainy,201,0.0573
6,Thunder,Heavy-Rain,192,0.2468
7,Heavy-Rain,Cold-Rainy,173,0.1014
8,Cold-Dry,Mild-Dry,139,0.0396
9,Thunder,Hot-Dry,138,0.1774
